# Transaction exceptions in the original workbook

Use a CPU and run from top to bottom. Inspect three overlapping sets of **original rows**: invoices beginning with `C`, quantities ≤ 0, and unit prices ≤ 0. Exact duplicates and missing customers remain in scope. The earlier five-rule audit simulated sequential exclusions. Here, every comparison starts from the full worksheet. All 17 output worksheets are built in the code cells below. For the zero-price investigation, look for “Which rows have a nonpositive unit price?” and “Invoice 550188”. The Excel output contains all qualifying rows and their source row numbers.

# 原始工作簿中的交易异常

使用 CPU，从上到下运行。核查三组可能重叠的**原始行**：发票号以 `C` 开头、数量 ≤ 0、单价 ≤ 0。完全重复行和缺少客户编号的行均参与统计。此前的五规则审计模拟了逐步排除记录；本次各项比较均从完整原表出发。17 张工作表的计算写在下方单元格中。核查零价问题时，可从「哪些记录的单价非正？」和「发票 550188」两节开始阅读。唯一的输出文件是一份 Excel 工作簿，内含可筛选的表格和原始行号。

## Input and output

In Colab, mount Drive and use the original `data/raw/Online Retail.xlsx`. If prompted, copy its path from the Files panel. A workbook under `data/raw/` identifies the project folder automatically. For a different layout, select the project's output folder. Locally, run from the repository root or from `docs/technical/`. Check both printed paths.

## 输入与输出

在 Colab 挂载 Drive，使用原始的 `data/raw/Online Retail.xlsx`。出现路径提示时，从左侧「文件」面板复制工作簿路径。如果它位于 `data/raw/` 下，代码会自动定位项目文件夹。其他目录结构需指定用于存放结果的项目文件夹。本地可从仓库根目录或 `docs/technical/` 运行。请核对打印的两条路径。

In [1]:
from pathlib import Path
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is None:
    project = Path.cwd()
    if project.parts[-2:] == ("docs", "technical"):
        project = project.parents[1]
    workbook = project / "data/raw/Online Retail.xlsx"
else:
    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/CS5481_Project2_OnlineRetail")
    workbook = project / "data/raw/Online Retail.xlsx"
    if not workbook.is_file():
        workbook = Path(input("Full workbook path / 工作簿完整路径：").strip())
        if workbook.parent.name == "raw" and workbook.parent.parent.name == "data":
            project = workbook.parents[2]
        else:
            project = Path(input("Drive project folder / Drive 项目文件夹完整路径：").strip())
            if not project.is_absolute() or not project.is_dir():
                raise ValueError("Select an existing project folder / 请选择已存在的项目文件夹")

if not workbook.is_file():
    raise FileNotFoundError(f"Workbook not found / 找不到工作簿：{workbook}")
output = project / "results/transaction_exceptions.xlsx"
print("Input / 输入：", workbook)
print("Output / 输出：", output)

Input / 输入： /workspace/online-retail-top10-recommendation/data/raw/Online Retail.xlsx
Output / 输出： /workspace/online-retail-top10-recommendation/results/transaction_exceptions.xlsx


## Verify the original eight fields

Check the workbook fingerprint, preserve the three identifiers as text, and keep the original Excel row number. The remaining cells run entirely inside this notebook.

## 核查原表的八个字段

先核对工作簿指纹，按文本读取三个编号列，并保留 Excel 原始行号。后面的分析代码均可在本 notebook 内直接运行。

In [2]:
import hashlib
import re
import pandas as pd

RAW_COLUMNS = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country",
]
with workbook.open("rb") as source:
    digest = hashlib.sha256(source.read()).hexdigest()
    if digest != "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d":
        raise ValueError("Check the original workbook / 请核对原始工作簿")
    source.seek(0)
    raw = pd.read_excel(source, sheet_name="Online Retail", engine="openpyxl",
                        dtype={c: "string" for c in ("InvoiceNo", "StockCode", "CustomerID")},
                        keep_default_na=False, na_values=[""])
assert raw.columns.tolist() == RAW_COLUMNS
raw.insert(0, "ExcelRow", range(2, len(raw) + 2))
tables = {}
print(f"Source rows / 原始行数：{len(raw):,}")

Source rows / 原始行数：541,909


## Count the sign combinations

From each original line derive the three flags and a signed line amount. Each row in `sign_combinations` groups lines by invoice prefix, sign of quantity, sign of unit price, and presence of a customer ID. The groups partition the worksheet. `rows` counts source rows, `invoices` counts distinct invoice numbers within that group, and `signed_gbp` sums quantity × unit price. One invoice may appear in several groups.

## 统计符号组合

由每一条原始记录计算三种标记和带符号的行金额。`sign_combinations` 按发票前缀、数量符号、单价符号和是否有客户编号分组，每条原始行恰好落入一个组合。`rows` 是原始行数，`invoices` 是组内不同发票号数，`signed_gbp` 为数量乘以单价后的带符号金额之和。同一张发票可以出现在多个组。

In [3]:
def _sign(values: pd.Series) -> pd.Series:
    signs = pd.Series("zero", index=values.index)
    signs.loc[values.gt(0)] = "positive"
    signs.loc[values.lt(0)] = "negative"
    return signs.mask(values.isna(), "missing")

lines = raw.assign(
    invoice_c=raw["InvoiceNo"].str.startswith("C", na=False),
    quantity_sign=_sign(raw["Quantity"]),
    price_sign=_sign(raw["UnitPrice"]),
    customer_present=raw["CustomerID"].notna(),
    amount_gbp=raw["Quantity"] * raw["UnitPrice"],
)
combinations = lines.groupby(
    ["invoice_c", "quantity_sign", "price_sign", "customer_present"],
    dropna=False, sort=True,
).agg(rows=("ExcelRow", "size"), invoices=("InvoiceNo", "nunique"),
      stock_codes=("StockCode", "nunique"), signed_gbp=("amount_gbp", "sum")).reset_index()
tables["sign_combinations"] = combinations
assert combinations["rows"].sum() == len(raw)
display(combinations)

,invoice_c,quantity_sign,price_sign,customer_present,rows,invoices,stock_codes,signed_gbp
0,False,negative,zero,False,1336,1336,1082,0.000
1,False,positive,negative,False,2,2,1,-22124.120
2,False,positive,positive,False,132220,1428,3408,1755276.640
3,False,positive,positive,True,397884,18532,3665,8911407.904
4,False,positive,zero,False,1139,785,654,0.000
5,False,positive,zero,True,40,34,35,0.000
6,True,negative,positive,False,383,182,161,-285470.400
7,True,negative,positive,True,8905,3654,1920,-611342.090


In [4]:
exception = lines.invoice_c | lines.Quantity.le(0) | lines.UnitPrice.le(0)
flagged = lines.loc[exception].copy()
priced = lines.loc[~lines.invoice_c & lines.Quantity.gt(0) & lines.UnitPrice.gt(0)]
invoice_counts = priced.groupby("InvoiceNo").size()
flagged["same_invoice_positive_priced_rows"] = flagged.InvoiceNo.map(invoice_counts).fillna(0).astype(int)
line_fields = ["ExcelRow", "InvoiceNo", "InvoiceDate", "StockCode", "Description",
               "Quantity", "UnitPrice", "CustomerID", "amount_gbp"]
print(f"Distinct flagged source rows / 命中任一条件的原表行数：{len(flagged):,}")

Distinct flagged source rows / 命中任一条件的原表行数：11,805


**Reading the output**

**`sign_combinations`** · The eight observed combinations of invoice prefix, quantity sign, price sign and customer-ID availability account for all 541,909 source rows exactly once. The union of C invoices, nonpositive quantities and nonpositive prices contains 11,805 distinct source rows.

**如何解读输出**

**`sign_combinations`** · 按发票前缀、数量符号、价格符号和客户编号有无分组，实际出现的八种组合恰好覆盖原表 541,909 行。C 发票、数量非正和单价非正三个条件的并集涉及 11,805 个原始行号。

## Which rows have a nonpositive unit price?

UCI defines `UnitPrice` as the product price per unit in sterling. It supplies no reason field explaining a zero price on every line. Filter the raw rows on this one field, then group by the sign of the price, the sign of the quantity, and the presence of a customer ID. There are 2,515 zero-price rows and two negative-price rows. The four groups in the output account for all 2,517 rows exactly once. `invoices` counts distinct invoice IDs within each group. [UCI Online Retail variable definitions](https://archive.ics.uci.edu/dataset/352/online+retail)

## 哪些记录的单价非正？

UCI 将 `UnitPrice` 定义为以英镑计价的商品单价。原表没有逐行解释零单价原因的字段。先按单价筛选原始记录，再按价格符号、数量符号和客户编号是否存在分组。原表中 2,515 行零价、2 行负价。输出中四组互斥，合计 2,517 行。`invoices` 统计各组内不同的发票编号数。[UCI Online Retail 字段定义](https://archive.ics.uci.edu/dataset/352/online+retail)

In [5]:
price_rows = lines.loc[lines["UnitPrice"].le(0)].copy()
price_rows["blank_description"] = price_rows["Description"].fillna("").str.strip().eq("")
price_groups = (
    price_rows.groupby(["price_sign", "quantity_sign", "customer_present"], dropna=False)
    .agg(rows=("ExcelRow", "size"), invoices=("InvoiceNo", "nunique"),
         blank_descriptions=("blank_description", "sum"))
    .reset_index()
)
assert price_groups["rows"].sum() == len(price_rows)
display(price_groups)

,price_sign,quantity_sign,customer_present,rows,invoices,blank_descriptions
0,negative,positive,False,2,2,0
1,zero,negative,False,1336,1336,862
2,zero,positive,False,1139,785,592
3,zero,positive,True,40,34,0


### What does the original wording support?

- **Zero price, negative quantity, no customer.** There are 1,336 rows, including 862 with a blank description. Literal descriptions include `damaged`, `samples`, and `Unsaleable, destroyed.`. They give different possible contexts for an inventory decrease.
- **Zero price, positive quantity, no customer.** There are 1,139 rows, including 592 with a blank description. Some entries say `found` or `adjustment`. A stock entry is plausible for those cases. A product name or blank description alone does not establish a reason.
- **Zero price, positive quantity, identified customer.** All 40 descriptions are present. Six rows have code `M` and description `Manual`. The other 34 have product descriptions. Their presence beside priced items is checked below.
- **Negative price, positive quantity, no customer.** Invoices `A563186` and `A563187` each record one unit at −£11,062.06. Both rows say `Adjust bad debt` and carry code `B`. Show both rows with the positive-price entry on `A563185` for comparison. These are accounting entries. Their signed line amounts by themselves do not establish cash settlement.

The next table counts **exact** selected descriptions, preserving case and punctuation. The examples cover only those literal strings.

### 原始文字能支持哪些解释？

- **零价、负数量、无客户编号。** 共 1,336 行，其中 862 行缺描述。原文包括 `damaged`（损坏）、`samples`（样品）、`Unsaleable, destroyed.`（无法出售，已销毁）。这些文字提示数量减少可能有不同背景。
- **零价、正数量、无客户编号。** 共 1,139 行，其中 592 行缺描述。部分原文为 `found`（找到）或 `adjustment`（调整），相应行可能记录库存操作。仅凭商品名称或空描述无法确定原因。
- **零价、正数量、有客户编号。** 40 行均有描述，其中 6 行的商品码为 `M`、描述为 `Manual`，其余 34 行是商品描述。下文核查同票中是否还有正常计价商品。
- **负价、正数量、无客户编号。** 发票 `A563186` 和 `A563187` 各记录 1 件，单价均为 −£11,062.06。两行均写有 `Adjust bad debt`（调整坏账），商品码均为 `B`。下方将两行与 `A563185` 上的正价条目并排展示。这些是账务记录，仅凭带符号的行金额无法确认实际资金结算。

下表按所选描述**精确匹配**，保留原始大小写和标点。计数只覆盖列出的文字。

In [6]:
selected_descriptions = ["damaged", "samples", "Unsaleable, destroyed.", "found", "adjustment", "Manual"]
exact_notes = price_rows.loc[price_rows["Description"].isin(selected_descriptions)]
display(
    exact_notes.groupby(["quantity_sign", "customer_present", "Description"])
    .size().rename("rows").reset_index()
)

debt_entries = lines.loc[
    lines["InvoiceNo"].isin(["A563185", "A563186", "A563187"]) & lines["StockCode"].eq("B"),
    ["ExcelRow", "InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "amount_gbp"],
]
display(debt_entries)

,quantity_sign,customer_present,Description,rows
0,negative,False,"Unsaleable, destroyed.",9
1,negative,False,adjustment,2
2,negative,False,damaged,42
3,negative,False,samples,1
4,positive,False,adjustment,14
5,positive,False,damaged,1
6,positive,False,found,25
7,positive,True,Manual,6


,ExcelRow,InvoiceNo,StockCode,Description,Quantity,UnitPrice,amount_gbp
299982,299984,A563185,B,Adjust bad debt,1,11062.06,11062.06
299983,299985,A563186,B,Adjust bad debt,1,-11062.06,-11062.06
299984,299986,A563187,B,Adjust bad debt,1,-11062.06,-11062.06


### Why does the final price filter account for 40 additional rows?

Recreate eligibility immediately before the price step on the **eight original fields**. A row must survive exact-duplicate removal, have a customer ID and a non-C invoice, and have positive quantity. Then apply `UnitPrice <= 0`. Comparing source row numbers verifies that the resulting 40 rows are exactly the identified, positive-quantity, zero-price group; both negative-price debt rows already failed the missing-ID rule.

For each of the 40 rows, look for another non-C, positive-quantity, positive-price line on the **same invoice**. The counts distinguish rows from invoices: the 40 rows span 34 invoices; 33 of the rows, on 30 invoices, have a priced companion. The other seven rows span four invoices that contain only zero-price lines. The detail table lists all 40 source rows. A complimentary item, promotion, or data-entry error needs additional evidence.

### 为什么最后的价格规则净排除 40 行？

使用**全部八个原始字段**复算价格步骤前的保留条件：排除完全重复行后，客户编号存在、发票号非 C、数量为正。再应用 `UnitPrice <= 0`。通过对比原始 Excel 行号，可以验证剩下的 40 行恰好是有客户编号、数量为正的零价组。两条负价坏账行已在缺失客户编号步骤排除。

对这 40 行逐行检查**同张发票**是否还包含非 C、正数量、正单价的其他记录。40 行分属 34 张发票；其中有 33 行与正常计价行同票，这 33 行分属 30 张发票。其余 7 行分属 4 张全票零价的发票。下方列出全部 40 行，可逐行核对。确认是否为赠品、促销或价格录入问题仍需其他证据。

In [7]:
duplicate = raw.duplicated(RAW_COLUMNS, keep="first")
before_price = (
    ~duplicate & raw["CustomerID"].notna()
    & ~raw["InvoiceNo"].str.startswith("C", na=False) & raw["Quantity"].gt(0)
)
final_price_exclusions = raw.loc[before_price & raw["UnitPrice"].le(0)]

price_evidence = flagged.loc[flagged["UnitPrice"].le(0)]
identified_zero = price_evidence.loc[
    price_evidence["UnitPrice"].eq(0) & price_evidence["Quantity"].gt(0)
    & price_evidence["CustomerID"].notna()
].copy()
assert set(final_price_exclusions["ExcelRow"]) == set(identified_zero["ExcelRow"])
print("Net rows removed at price step / 价格步骤净排除行数：", len(final_price_exclusions))

identified_zero["manual_entry"] = identified_zero["StockCode"].eq("M") & identified_zero["Description"].eq("Manual")
identified_zero["priced_companion"] = identified_zero["same_invoice_positive_priced_rows"].gt(0)
display(identified_zero.groupby("manual_entry").size().rename("rows").reset_index())
display(identified_zero.groupby("priced_companion").agg(
    zero_price_rows=("ExcelRow", "size"), distinct_invoices=("InvoiceNo", "nunique")
).reset_index())

display(identified_zero[[
    "ExcelRow", "InvoiceNo", "StockCode", "Description", "Quantity", "CustomerID",
    "same_invoice_positive_priced_rows", "priced_companion",
]].sort_values("ExcelRow"))

Net rows removed at price step / 价格步骤净排除行数： 40


,manual_entry,rows
0,False,34
1,True,6


,priced_companion,zero_price_rows,distinct_invoices
0,False,7,4
1,True,33,30


,ExcelRow,InvoiceNo,StockCode,Description,Quantity,CustomerID,same_invoice_positive_priced_rows,priced_companion
9302,9304,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,12647,16,True
33576,33578,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,16560,20,True
40089,40091,539722,22423,REGENCY CAKESTAND 3 TIER,10,14911,28,True
47068,47070,540372,22090,PAPER BUNTING RETROSPOT,24,13081,169,True
47070,47072,540372,22553,PLASTERS IN TIN SKULLS,24,13081,169,True
56674,56676,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,15107,1,True
86789,86791,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,17560,0,False
130188,130190,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,13239,21,True
139453,139455,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,13113,6,True
145208,145210,548871,22162,HEART GARLAND RUSTIC PADDED,2,14410,6,True


### Invoice 550188: one SKU at two prices

Customer 12457 has two lines for SKU 22636 on the same invoice. The eight units at £7.65 yield £61.20. The other unit has a recorded price of £0.00. For a Top-10 target defined as **products purchased at a positive price**, exclude the zero-price line while keeping the eight priced units and one distinct purchased SKU in this basket. Removing the whole invoice or permanently banning the SKU would erase the priced purchase. A complimentary unit is one possible explanation. The invoice records the price but gives no reason for it.

### 发票 550188：同一商品出现两种单价

客户 12457 在同张发票上有两行商品码 22636。8 件商品按 £7.65 计价，行金额 £61.20；另 1 件的记录单价为 £0.00。若 Top-10 的目标定义为**以正价购买的商品**，则排除零价行，同时保留 8 件正常计价商品，该购物篮中这个商品编码计 1 次。删除整张发票或永久排除这个商品码都会丢失正常计价的购买记录。这与随单赠送的情形相符，发票只记录了价格，未注明赠送原因。

In [8]:
invoice_item = raw.loc[raw["InvoiceNo"].eq("550188") & raw["StockCode"].eq("22636")].copy()
invoice_item["line_gbp"] = invoice_item["Quantity"] * invoice_item["UnitPrice"]
display(invoice_item[[
    "ExcelRow", "CustomerID", "InvoiceNo", "InvoiceDate", "StockCode",
    "Description", "Quantity", "UnitPrice", "line_gbp",
]])
paid_item = invoice_item.loc[invoice_item["Quantity"].gt(0) & invoice_item["UnitPrice"].gt(0)]
display(paid_item.groupby(["CustomerID", "InvoiceNo"], as_index=False).agg(
    purchased_skus=("StockCode", "nunique"), priced_units=("Quantity", "sum"),
    priced_amount_gbp=("line_gbp", "sum")
))

,ExcelRow,CustomerID,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,line_gbp
157029,157031,12457,550188,2011-04-14 18:57:00,22636,CHILDS BREAKFAST SET CIRCUS PARADE,8,7.65,61.2
157042,157044,12457,550188,2011-04-14 18:57:00,22636,CHILDS BREAKFAST SET CIRCUS PARADE,1,0.00,0.0


,CustomerID,InvoiceNo,purchased_skus,priced_units,priced_amount_gbp
0,12457,550188,1,8,61.2


## Missing fields and competing filters

Read `missingness` before deciding how a field can be used. A customer ID identifies a person. Averaging customer numbers or inventing an ID would combine unrelated histories. Keep unidentified rows for this raw-data investigation and count them separately. The project excludes them when it constructs personalized purchase histories. A row with a valid StockCode can still identify a product when Description is missing. Use the code as its display label when a suitable description is unavailable. The invoice table lets you inspect which invoice numbers are affected by absent customer IDs.

The `filter_attribution` table revisits the earlier hypothetical five-rule filter in its specified order. For the 10,624 rows with nonpositive quantity, 37 are first accounted for by duplicates, 1,715 by missing customer ID, and 8,872 by the C-invoice flag. Thus quantity contributes zero additional removals at its step. The final price rule accounts for 40 additional rows, all with zero unit price. These figures describe overlap and rule order. All original rows still feed the exception tables below.

All 135,080 missing customer IDs occur on 3,710 invoices. Across the complete sheet, none of those invoices contains a known customer on another line. All 1,454 missing descriptions also lack a customer ID. Quantity and UnitPrice have no missing values. A zero price is an observed value. Lecture 3, slide 10, lists imputation as one possible response to missing data. Here a mean cannot identify a buyer or establish a paid price. [Lecture 3](../../resource/Lecture_3.pdf)

## 缺失字段与相互重叠的规则

先查看 `missingness`，再判断每个字段如何用于分析。客户编号用于识别客户，对编号取均值或虚构编号会把无关历史混在一起。本次原始数据核查保留无编号行并单独计数，项目在构建个性化购买历史时才剔除这些行。如果 StockCode 有效，即使缺少 Description 也能识别商品；没有合适的商品名称时，可直接用商品编码展示。发票表可用来定位哪些发票号涉及客户编号缺失。

`filter_attribution` 按此前假设的五规则顺序追溯净排除量。在 10,624 条非正数量行中，37 条首先归入完全重复，1,715 条首先归入客户编号缺失，8,872 条首先归入 C 发票。因此数量规则本身新增排除 0 行。最后的价格规则新增排除 40 行，均为零单价。以上是规则重叠及顺序产生的归因，后续异常表仍使用所有原始行。

缺客户编号的 135,080 行涉及 3,710 张发票。回查整张原表，这些发票的其他明细也都没有已知客户编号。缺商品描述的 1,454 行全部同时缺客户编号。数量和单价都没有空值，零价是原表实际记录的值。Lecture 3 第 10 页列出多种缺失值处理方式；在此，对编号求均值无法识别买家，用均价替代零也无法证明付费。[Lecture 3](../../resource/Lecture_3.pdf)

### Missing fields across the original worksheet · `missingness`

For each of the eight original fields, count missing rows, the percentage of all 541,909 rows they represent, and the number of distinct affected invoices. The last column counts the intersection with missing customer IDs. Of the 135,080 rows missing a customer ID, 1,454 also have a missing description. A row can appear in several field counts.

### 原始工作表的字段缺失情况 · `missingness`

针对八个原始字段分别统计缺失行数、占全部 541,909 行的比例，以及涉及的不同发票数。最后一列统计该字段和客户编号同时缺失的行数。135,080 行缺少客户编号，其中 1,454 行也缺少商品描述。一行可能计入多个字段的缺失数。

In [9]:
missing_customer = raw["CustomerID"].isna()
missing_records = []
for field in RAW_COLUMNS:
    missing = raw[field].isna()
    missing_records.append({
        "field": field,
        "raw_rows": len(raw),
        "missing_rows": int(missing.sum()),
        "missing_pct": 100 * missing.mean(),
        "invoices_with_missing": raw.loc[missing, "InvoiceNo"].nunique(),
        "missing_with_missing_customer_id": int((missing & missing_customer).sum()),
    })

tables["missingness"] = pd.DataFrame.from_records(missing_records)
display(tables["missingness"])

,field,raw_rows,missing_rows,missing_pct,invoices_with_missing,missing_with_missing_customer_id
0,InvoiceNo,541909,0,0.000000,0,0
1,StockCode,541909,0,0.000000,0,0
2,Description,541909,1454,0.268311,1454,1454
3,Quantity,541909,0,0.000000,0,0
4,InvoiceDate,541909,0,0.000000,0,0
5,UnitPrice,541909,0,0.000000,0,0
6,CustomerID,541909,135080,24.926694,3710,135080
7,Country,541909,0,0.000000,0,0


### Can another line on the same invoice supply a customer ID? · `customer_id_invoices`

Find every invoice with at least one missing customer ID. Then retrieve **all** of its lines and count the distinct known IDs. Inspect the first ten invoices and the distribution across the full table below. All 3,710 affected invoices have zero known customer IDs on their other lines. This lookup therefore supplies no customer ID to fill in. The worksheet row numbers let you check each case in Excel.

### 同一张发票的其他行能否补出客户编号？ · `customer_id_invoices`

先找出至少有一行缺少客户编号的发票，再取回这些发票的**所有行**，统计每张发票上已知的不同客户编号数。下方展示前十张发票，以及完整表中已知编号数的分布。3,710 张受影响的发票上均找不到任何已知客户编号，因此无法从同票的其他行补值。可以用工作表行号回到 Excel 核查。

In [10]:
affected_invoices = raw.loc[missing_customer, "InvoiceNo"].dropna().unique()
affected_lines = raw.loc[
    raw["InvoiceNo"].isin(affected_invoices), ["InvoiceNo", "CustomerID", "ExcelRow"]
].copy()
affected_lines["missing_customer"] = affected_lines["CustomerID"].isna()
affected_lines["missing_excel_row"] = affected_lines["ExcelRow"].where(affected_lines["missing_customer"])
missing_invoices = affected_lines.groupby("InvoiceNo", sort=False).agg(
    first_excel_row=("ExcelRow", "min"),
    first_missing_excel_row=("missing_excel_row", "min"),
    total_rows=("ExcelRow", "size"),
    missing_customer_rows=("missing_customer", "sum"),
    known_customer_ids=("CustomerID", "nunique"),
).reset_index().sort_values("first_excel_row", ignore_index=True)
missing_invoices["first_missing_excel_row"] = missing_invoices["first_missing_excel_row"].astype("int64")

tables["customer_id_invoices"] = missing_invoices
display(missing_invoices.head(10))
display(missing_invoices["known_customer_ids"].value_counts().rename("invoices").to_frame())

,InvoiceNo,first_excel_row,first_missing_excel_row,total_rows,missing_customer_rows,known_customer_ids
0,536414,624,624,1,1,0
1,536544,1445,1445,527,527,0
2,536545,1972,1972,1,1,0
3,536546,1973,1973,1,1,0
4,536547,1974,1974,1,1,0
5,536549,1989,1989,1,1,0
6,536550,1990,1990,1,1,0
7,536552,2026,2026,1,1,0
8,536553,2027,2027,1,1,0
9,536554,2028,2028,1,1,0


,invoices
known_customer_ids,
0,3710


### Which filter first accounts for each affected row? · `filter_attribution`

Apply the proposed purchase-sample rules in order to a temporary classification. Detect duplicates across the eight original fields and keep the first occurrence. Then flag a missing customer ID, a C-prefixed invoice, a nonpositive quantity, and a nonpositive unit price in that order. Each row contributes once within a scope. Scopes such as C-prefixed invoices and negative quantities can overlap. A zero under a later filter means earlier filters have already accounted for all relevant rows in that scope. All original rows remain available for inspection.

### 各类明细最先命中哪一条过滤规则？ · `filter_attribution`

按计划中的购买样本规则对原表临时分类。先基于八个原始字段判断完全重复并保留首次出现的行，再依次判断缺客户编号、发票号以 C 开头、数量非正和单价非正。一个范围内每行只计一次。不同范围之间可以重叠，例如 C 开头的发票与负数量明细。靠后的规则计数为零时，要检查前面的规则是否已经计入该范围内的所有相关行。原始明细仍可随时回查。

In [11]:
purchase_flags = {
    "exact_duplicate": raw.duplicated(subset=RAW_COLUMNS, keep="first"),
    "missing_customer_id": missing_customer,
    "cancelled_invoice": raw["InvoiceNo"].astype("string").str.startswith("C", na=False),
    "nonpositive_quantity": raw["Quantity"].le(0).fillna(False),
    "nonpositive_unit_price": raw["UnitPrice"].le(0).fillna(False),
}
first_rule = pd.Series("retained", index=raw.index)
for rule, condition in purchase_flags.items():
    first_rule.loc[first_rule.eq("retained") & condition] = rule

attribution_scopes = {
    "all_rows": pd.Series(True, index=raw.index),
    "C_invoices": purchase_flags["cancelled_invoice"],
    "Q_nonpositive": purchase_flags["nonpositive_quantity"],
    "P_nonpositive": purchase_flags["nonpositive_unit_price"],
    "zero_quantity": raw["Quantity"].eq(0),
    "negative_quantity": raw["Quantity"].lt(0),
    "zero_price": raw["UnitPrice"].eq(0),
    "negative_price": raw["UnitPrice"].lt(0),
    "missing_customer_id": missing_customer,
    "missing_description": raw["Description"].isna(),
}
attribution_rows = []
for scope, members in attribution_scopes.items():
    counts = first_rule.loc[members.fillna(False)].value_counts()
    row = {"scope": scope, "raw_rows": int(counts.sum())}
    for rule in [*purchase_flags, "retained"]:
        row[rule] = int(counts.get(rule, 0))
    attribution_rows.append(row)

tables["filter_attribution"] = pd.DataFrame.from_records(attribution_rows)
display(tables["filter_attribution"])

,scope,raw_rows,exact_duplicate,missing_customer_id,cancelled_invoice,nonpositive_quantity,nonpositive_unit_price,retained
0,all_rows,541909,5268,135037,8872,0,40,392692
1,C_invoices,9288,37,379,8872,0,0,0
2,Q_nonpositive,10624,37,1715,8872,0,0,0
3,P_nonpositive,2517,5,2472,0,0,40,0
4,zero_quantity,0,0,0,0,0,0,0
5,negative_quantity,10624,37,1715,8872,0,0,0
6,zero_price,2515,5,2470,0,0,40,0
7,negative_price,2,0,2,0,0,0,0
8,missing_customer_id,135080,43,135037,0,0,0,0
9,missing_description,1454,0,1454,0,0,0,0


## Negative quantities on non-C invoices and earlier positive quantities

Start with `nonC_negative`. The 1,336 lines each occupy a single-line invoice, have zero unit price and lack a customer ID. Of these, 862 also lack a description. Use `history_summary` to compare earlier positive-quantity lines for the exact invoice, invoice + SKU, SKU alone, and customer + SKU. Earlier means a strictly earlier timestamp. The positive-quantity lookups include every earlier row with Quantity > 0 regardless of price or missing ID. The stronger customer/SKU comparison also requires a known customer and a non-C positive-price purchase line. A blank customer match is unavailable evidence. A previous occurrence of the SKU shows the item was recorded before. Matching the customer, price, quantity magnitude, and time order provides stronger evidence of a possible reversal. Even that match alone does not verify a physical return or refund.

For the 1,336 lines, the exact-invoice and exact-invoice-plus-SKU searches each find zero earlier positive lines. The weaker SKU-only search succeeds on 1,179 lines. For the 9,288 C lines, removing the C and looking for that invoice number plus SKU finds zero earlier positive matches. A different search finds 2,816 C lines with at least one earlier non-C, positive-price line for the same customer, SKU, unit price and opposite quantity. These are candidate lines, potentially several per credit. The workbook contains no allocation of each credit to an original sale.

## 非 C 发票的负数量与此前的正数量记录

从 `nonC_negative` 入手。这 1,336 行每行各占一张仅有一行的发票，单价为零、客户编号缺失，其中 862 行还缺少商品描述。通过 `history_summary` 分别比较同发票、同发票与商品、同商品，以及同客户与商品的早先正数量记录。“早先”要求时间戳严格早于当前行。按正数量回查时，单价为零或缺客户编号的早期正数量行也参与。核对更强的同客户、同商品候选时，另要求客户编号已知，且候选购买行单价为正、发票号不以 C 开头。缺少客户编号时，客户匹配结果为空，表示无法核对。商品编码曾出现，只说明原表中此前记录过这个商品。如果客户、单价、数量绝对值和时间顺序都吻合，则更支持可能存在冲销；仅凭匹配仍无法确认实物退回或款项退回。

对 1,336 行按原发票号、以及按原发票号加商品编码查找，两个口径都没有找到此前的正数量行。放宽到仅按商品编码匹配，则有 1,179 行找到早期记录。对 9,288 条 C 明细去掉 C 前缀后再按发票号加商品回查，也没有早期匹配。换一种方法，按同一客户、同一商品、同一单价、数量恰好相反四个条件查找，2,816 行有更早的非 C、正价候选购买行。一条贷项可能对应多个候选，原表没有给出逐笔核销关系。

## Which transactions were already on record?

Use each flagged line as a query against the original worksheet. For an invoice, SKU, or customer–SKU pair, count positive-quantity **lines**, retaining repeated entries. Sort historical lines by timestamp and then by Excel row. Within each group of matching identifiers, `cumcount() + 1` assigns the number of lines seen so far. A backward `merge_asof` looks up the last count **strictly before** the query. A line with the same timestamp is ineligible, even if it occurs earlier on the worksheet.

A count of **0** means the necessary identifiers and timestamp are present and no earlier line matches. **NA** means a required identifier or timestamp is missing, or that a C-invoice diagnostic does not apply. The full-sheet invoice count includes the flagged row and all matching invoice lines, regardless of date.

## 这条明细出现之前，工作簿中已有怎样的交易记录？

用每条标记明细查询原始工作表。按发票、商品或客户与商品的组合统计数量为正的**明细行数**，保留原表中的重复行。先按交易时间排序，同一时间再按 Excel 行号排序。在标识符相同的每个组内，`cumcount() + 1` 给出截至每一行的累计行数。`merge_asof` 向前查找**严格早于**标记时间的最后一个累计值。同一时间的交易即使在表中的行号较小，也不能作为更早的记录。

计数为 **0** 表示标识符和时间齐全，但没有找到符合条件的更早明细。**NA** 表示缺少比较所需的标识符或时间，或该行不适用 C 发票专用的比较。整表发票行数包含标记行本身以及所有同发票号的行，不受时间限制。

In [12]:
def strictly_earlier_match(history, queries, keys, columns):
    right = history.dropna(subset=keys)[[*keys, "InvoiceDate", *columns]]
    left = queries.dropna(subset=keys)[[*keys, "InvoiceDate", "_position"]]
    matched = pd.merge_asof(
        left, right, on="InvoiceDate", by=keys,
        allow_exact_matches=False, direction="backward",
    )
    return matched.set_index("_position")[columns]


def earlier_line_counts(history, queries, keys):
    running = history[[*keys, "InvoiceDate"]].copy()
    running["_count"] = running.groupby(keys, sort=False).cumcount() + 1
    matched = strictly_earlier_match(running, queries, keys, ["_count"])
    return matched["_count"].fillna(0).astype("Int64")

### Compare the invoice, the item and the customer

Start with the exact invoice number, including any leading C. Then compare the invoice–SKU pair, the SKU across the whole sheet, and the customer–SKU pair. For these counts, a positive quantity is sufficient. To show a plausible purchase, narrow the search to lines with a positive price and a non-C invoice number for the same customer and SKU. Record the latest eligible time and its Excel row. If two candidates share that time, select the higher Excel row to make the example reproducible.

### 按发票、商品和客户核查

先查完整发票号（保留开头的 C），再分别按发票与商品、全表中的商品、客户与商品核查。上述计数只要求历史行数量为正。为展示一条可能的购买记录，再缩小范围：要求客户和商品均相同、单价为正、发票号存在且不以 C 开头。记录时间最近的符合条件的行。若最近时间相同，取 Excel 行号较大的一条，保证每次展示相同的示例。

In [13]:
history_fields = ["InvoiceNo", "StockCode", "CustomerID", "InvoiceDate", "Quantity", "UnitPrice"]
history_queries = flagged[history_fields].reset_index(drop=True)
full_invoice_counts = raw["InvoiceNo"].value_counts()
history_evidence = pd.DataFrame({
    "invoice_rows_full_sheet": history_queries["InvoiceNo"].map(full_invoice_counts).astype("Int64")
})
history_queries["_position"] = history_queries.index
dated_queries = history_queries.dropna(subset=["InvoiceDate"]).sort_values("InvoiceDate")
positive_history = raw.loc[
    raw["Quantity"].gt(0) & raw["InvoiceDate"].notna(), [*history_fields, "ExcelRow"]
].sort_values(["InvoiceDate", "ExcelRow"])

history_keys = {
    "earlier_positive_invoice_rows": ["InvoiceNo"],
    "earlier_positive_invoice_sku_rows": ["InvoiceNo", "StockCode"],
    "earlier_positive_sku_rows": ["StockCode"],
    "earlier_positive_customer_sku_rows": ["CustomerID", "StockCode"],
}
for field, keys in history_keys.items():
    history_evidence[field] = earlier_line_counts(positive_history, dated_queries, keys)

In [14]:
plausible_purchases = positive_history.loc[
    positive_history["UnitPrice"].gt(0)
    & positive_history["InvoiceNo"].notna()
    & ~positive_history["InvoiceNo"].str.startswith("C", na=False)
].copy()
plausible_purchases["_purchase_date"] = plausible_purchases["InvoiceDate"]
latest_purchase = strictly_earlier_match(
    plausible_purchases, dated_queries, ["CustomerID", "StockCode"],
    ["ExcelRow", "_purchase_date"],
)
history_evidence["latest_plausible_customer_sku_excel_row"] = latest_purchase["ExcelRow"].astype("Int64")
history_evidence["latest_plausible_customer_sku_date"] = latest_purchase["_purchase_date"]

### Two additional comparisons for C-prefixed invoices

First remove one leading C and count earlier positive-quantity lines with that invoice number and the same SKU. A match would show a coincidence of identifiers. Independently, look for earlier plausible purchase lines with the same customer, SKU and exact unit price whose quantity equals the cancellation quantity with its sign reversed. This second comparison can find candidates on a different invoice. A candidate helps trace the records. The worksheet alone cannot establish which invoice a credit refers to or whether goods physically came back.

### C 发票的两项补充核查

第一项只去掉发票号开头的一个 C，再统计发票号与该编号相同、商品相同且数量为正的更早明细。若有匹配，只说明标识符吻合。第二项独立比较客户、商品、单价和数量：更早的可能购买记录必须有完全相同的客户、商品和单价，购买数量必须等于冲销行数量的相反数。这项核查可以找到不同发票上的候选行，供回查原始明细。仅凭这些字段无法确认冲销所指的原发票或货物是否实际退回。

In [15]:
c_queries = dated_queries.loc[dated_queries["InvoiceNo"].str.startswith("C", na=False)].copy()
c_queries["InvoiceNo"] = c_queries["InvoiceNo"].str.slice(1).replace("", pd.NA)
history_evidence["earlier_positive_stripped_invoice_sku_rows"] = earlier_line_counts(
    positive_history, c_queries, ["InvoiceNo", "StockCode"],
)
c_queries["Quantity"] = -c_queries["Quantity"]
history_evidence["earlier_matching_customer_sku_price_opposite_quantity_rows"] = earlier_line_counts(
    plausible_purchases, c_queries, ["CustomerID", "StockCode", "UnitPrice", "Quantity"],
)

# ExcelRow identifies the same source line across reruns. / Excel 行号始终指向同一条原始明细。
evidence_by_row = history_evidence.set_axis(flagged["ExcelRow"])
flagged = flagged.drop(columns=history_evidence.columns, errors="ignore").join(
    evidence_by_row, on="ExcelRow", validate="one_to_one",
)

## Annotate the lines with literal evidence

The following rules retain the words that match and the exact special code. A generic product name usually has no matching signal. The original description stays alongside each annotation.

## 给明细标记可回查的字面证据

下方规则记录命中的原始词语和特殊编码。普通商品名称常常没有关键词命中；每个标记旁都保留原始描述供复查。

In [16]:
# Keep the text exactly as written for the evidence column. Word boundaries prevent
# matches such as "count" in "country". A standalone "check" is a note, whereas
# "BROWN CHECK CAT DOORSTOP" describes a product.
_NOTE_RULES = {
    "damage_or_disposal": r"\b(?:damages?|damaged|breakages?|broken|faulty|smashed|crushed|cracked|destroyed|unsaleable|throw(?:n)? away|mouldy)\b",
    "missing_or_lost": r"\b(?:missing|lost|can't find|not rcvd)\b",
    "stock_adjustment_or_count": (
        r"\b(?:adjust(?:ment)?(?!\s+bad debt)|counted|stock check|incorrect stock entry|found)\b"
        r"|^\s*check(?=[\s?.!]*$)"
    ),
    "set_or_unit_sale_wording": (
        r"\bsold (?:as (?:sets?|1)|sets?)(?:/\d+)?"
        r"(?: (?:on|by) (?:dotcom|amazon)(?: and amazon)?)?\b"
    ),
    "item_code_or_label_discrepancy": (
        r"\b(?:(?:code|label) mix[- ]up|wrong(?:ly)? (?:coded|marked|mrked|code|barcode)"
        r"|came coded as|barcode problem)\b"
    ),
    "sample_or_promotion": r"\b(?:samples?|promotion(?:al)?|given away|free sample)\b",
    "fee_or_charge": r"\b(?:postage|carriage|bank charges|amazon fee|CRUK commission)\b",
    "discount": r"\bdiscount\b",
    "manual": r"^\s*manual\s*$",
    "debt": r"\bbad debt\b",
    "credit_note_wording": r"\b(?:credit|credits|credited|creditted)\b",
}
_NOTE_PATTERNS = {key: re.compile(expr, re.IGNORECASE) for key, expr in _NOTE_RULES.items()}
# These exact, case-sensitive codes have corresponding workbook descriptions.
_SPECIAL_CODES = {
    "S": "sample", "D": "discount", "B": "bad_debt", "M": "manual", "m": "manual",
    "POST": "postage", "DOT": "dotcom_postage", "C2": "carriage",
    "AMAZONFEE": "amazon_fee", "BANK CHARGES": "bank_charges", "CRUK": "commission",
}
# Workbook invoice identifiers have six digits, sometimes with a literal C or A
# prefix. Short numbers (e.g. "see 47") and longer alphanumeric IDs are not links.
_INVOICE_TOKEN = re.compile(r"\b[AC]?[0-9]{6}\b")

In [17]:
def _note_matches(text):
    groups, terms = [], []
    for name, pattern in _NOTE_PATTERNS.items():
        matches = pattern.findall(text)
        if matches:
            groups.append(name)
            terms.extend(match.strip() for match in matches)
    status = "blank" if not text.strip() else "unknown"
    return "|".join(groups) or status, "|".join(dict.fromkeys(terms))


def _mentioned_invoices(text):
    return list(dict.fromkeys(_INVOICE_TOKEN.findall(text)))

def description_evidence(raw: pd.DataFrame, flagged: pd.DataFrame) -> pd.DataFrame:
    """Return four evidence strings on flagged.index in the same order.

    Look up the original Description and StockCode by unique worksheet ExcelRow.
    Multiple signals, exact matched substrings, or distinct invoice tokens are
    joined by '|'. Literal text matches ignore case. Special codes match exactly.
    'blank' means missing/whitespace text. 'unknown' means no recognized text
    pattern; this also covers ordinary product names. An empty code signal or
    invoice list means no match. A note or code never establishes a return.
    """
    rows = raw[["ExcelRow", "Description", "StockCode"]].set_index("ExcelRow").loc[flagged["ExcelRow"]]
    notes = rows["Description"].astype("string").fillna("")
    matches = {note: _note_matches(note) for note in notes.unique()}
    references = {note: "|".join(_mentioned_invoices(note)) for note in matches}
    result = pd.DataFrame({
        "description_note_signals": notes.map({k: v[0] for k, v in matches.items()}),
        "description_matched_terms": notes.map({k: v[1] for k, v in matches.items()}),
        "special_code_signal": rows["StockCode"].map(_SPECIAL_CODES).fillna(""),
        "mentioned_invoice_ids": notes.map(references),
    })
    result.index = flagged.index
    return result

flagged = flagged.join(description_evidence(raw, flagged))

## Inspect the three sets side by side

The code prints the full size of each set and displays the first six rows in worksheet order. They are examples, not a statistical sample. `ExcelRow` points to the original worksheet including the header. A C-prefix is the dataset's cancellation indicator. A negative quantity records a decrease and a zero price gives a zero recorded amount regardless of quantity. Read the line description and invoice context before attaching a reason. Look at zero and negative values separately. In this workbook no quantity equals zero, 2,515 rows have zero unit price and two have negative unit price. The full sets, with additional evidence columns, are written to Excel.

## 并列检查三组记录

代码打印各组完整行数，并按工作表顺序展示前六行，便于查看实例。这里采用顺序展示，不能据此估计比例。`ExcelRow` 对应原始工作表的行号，首行为表头。C 前缀是数据集的取消标记。负数量记录数量减少，单价为零时，无论数量多少，行金额均为零。解释原因时还要查看描述与同票记录。数量、单价的零值和负值分别统计。该工作簿中数量为零有 0 行，单价为零有 2,515 行，单价为负有 2 行。完整记录及更多证据列均写入 Excel。

In [18]:
views = {
    "C_invoices": flagged.loc[flagged.invoice_c],
    "Q_nonpositive": flagged.loc[flagged.Quantity.le(0)],
    "P_nonpositive": flagged.loc[flagged.UnitPrice.le(0)],
    "nonC_negative": flagged.loc[~flagged.invoice_c & flagged.Quantity.lt(0)],
}
tables.update(views)
for name, group in views.items():
    print(f"{name} · {len(group):,} rows / 行")
    display(group[line_fields].head(6))

C_invoices · 9,288 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
141,143,C536379,2010-12-01 09:41:00,D,Discount,-1,27.50,14527,-27.50
154,156,C536383,2010-12-01 09:49:00,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,15311,-4.65
235,237,C536391,2010-12-01 10:24:00,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,17548,-19.80
236,238,C536391,2010-12-01 10:24:00,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,17548,-6.96
237,239,C536391,2010-12-01 10:24:00,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,17548,-6.96
238,240,C536391,2010-12-01 10:24:00,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,17548,-6.96


Q_nonpositive · 10,624 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
141,143,C536379,2010-12-01 09:41:00,D,Discount,-1,27.50,14527,-27.50
154,156,C536383,2010-12-01 09:49:00,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,15311,-4.65
235,237,C536391,2010-12-01 10:24:00,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,17548,-19.80
236,238,C536391,2010-12-01 10:24:00,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,17548,-6.96
237,239,C536391,2010-12-01 10:24:00,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,17548,-6.96
238,240,C536391,2010-12-01 10:24:00,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,17548,-6.96


P_nonpositive · 2,517 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
622,624,536414,2010-12-01 11:52:00,22139,NaN,56,0.0,<NA>,0.0
1970,1972,536545,2010-12-01 14:32:00,21134,NaN,1,0.0,<NA>,0.0
1971,1973,536546,2010-12-01 14:33:00,22145,NaN,1,0.0,<NA>,0.0
1972,1974,536547,2010-12-01 14:33:00,37509,NaN,1,0.0,<NA>,0.0
1987,1989,536549,2010-12-01 14:34:00,85226A,NaN,1,0.0,<NA>,0.0
1988,1990,536550,2010-12-01 14:34:00,85044,NaN,1,0.0,<NA>,0.0


nonC_negative · 1,336 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
2406,2408,536589,2010-12-01 16:50:00,21777,NaN,-10,0.0,<NA>,-0.0
4347,4349,536764,2010-12-02 14:42:00,84952C,NaN,-38,0.0,<NA>,-0.0
7188,7190,536996,2010-12-03 15:30:00,22712,NaN,-20,0.0,<NA>,-0.0
7189,7191,536997,2010-12-03 15:30:00,22028,NaN,-20,0.0,<NA>,-0.0
7190,7192,536998,2010-12-03 15:30:00,85067,NaN,-6,0.0,<NA>,-0.0
7192,7194,537000,2010-12-03 15:32:00,21414,NaN,-22,0.0,<NA>,-0.0


**Reading the output**

**`C_invoices`** · All 9,288 lines on the 3,836 C-prefixed invoices have negative quantities and positive unit prices. They include 77 discount lines (StockCode D), so the C prefix alone cannot establish that a customer physically returned an item.

**`Q_nonpositive`** · All 10,624 lines with Quantity ≤ 0 are strictly negative. They split exactly into 9,288 C-invoice lines and 1,336 non-C lines.

**`P_nonpositive`** · Of 2,517 lines with UnitPrice ≤ 0, 2,515 have zero recorded price. The two negative-price lines are A563186 and A563187, each with quantity 1, unit price −£11,062.06 and the description “Adjust bad debt”.

**`nonC_negative`** · All 1,336 non-C negative-quantity lines are on single-line invoices with zero unit price and missing customer ID. An earlier positive-quantity line exists for the same SKU in 1,179 cases, but there is no customer ID to link it to a buyer.

**如何解读输出**

**`C_invoices`** · 3,836 张 C 发票的 9,288 行均为负数量、正单价，其中包括 77 行折扣记录（StockCode 为 D）。仅凭 C 前缀无法认定客户实际退回了商品。

**`Q_nonpositive`** · Quantity ≤ 0 的 10,624 行全部为负数。按发票前缀划分，恰好是 9,288 行 C 发票记录和 1,336 行非 C 记录。

**`P_nonpositive`** · UnitPrice ≤ 0 的 2,517 行中，2,515 行记录的单价为零。仅 A563186 和 A563187 两行单价为负，两行均为数量 1、单价 −£11,062.06，描述为“Adjust bad debt”（坏账调整）。

**`nonC_negative`** · 全部 1,336 行非 C 负数量记录均为单行发票，单价为零且没有客户编号。其中 1,179 行的商品能找到较早的正数量记录，但缺少客户编号，无法据此关联买家。

## Summarize historical matches and note wording

Build `history_summary` and `note_summary` from the groups below. One source row can appear in several scopes. Count matching rows and show the combinations of literal note and stock-code signals.

## 按筛选范围汇总历史匹配与备注

按下方的核查范围计算 `history_summary` 和 `note_summary`，同一原始行可以进入多个范围。分别统计有历史匹配的明细行数，以及各种备注线索与商品码的组合。

In [19]:
scopes = {
    "any_exception": flagged,
    "C_invoices": views["C_invoices"],
    "nonC_negative": views["nonC_negative"],
    "zero_qty": flagged.loc[flagged.Quantity.eq(0)],
    "positive_qty_zero_price": flagged.loc[flagged.Quantity.gt(0) & flagged.UnitPrice.eq(0)],
    "negative_price": flagged.loc[flagged.UnitPrice.lt(0)],
}

In [20]:
history_rows = []
for scope, group in scopes.items():
    history_rows.append({
        "cohort": scope, "rows": len(group), "distinct_invoices": group.InvoiceNo.nunique(),
        "singleton_invoice_rows": group.invoice_rows_full_sheet.eq(1).sum(),
        "prior_same_invoice_rows_matched": group.earlier_positive_invoice_rows.gt(0).sum(),
        "prior_same_invoice_sku_rows_matched": group.earlier_positive_invoice_sku_rows.gt(0).sum(),
        "prior_sku_rows_matched": group.earlier_positive_sku_rows.gt(0).sum(),
        "customer_sku_comparable_rows": group.earlier_positive_customer_sku_rows.notna().sum(),
        "prior_customer_sku_rows_matched": group.earlier_positive_customer_sku_rows.gt(0).sum(),
        "prior_stripped_invoice_sku_rows_matched": group.earlier_positive_stripped_invoice_sku_rows.gt(0).sum(),
        "prior_customer_sku_price_opposite_units_rows_matched":
            group.earlier_matching_customer_sku_price_opposite_quantity_rows.gt(0).sum(),
        "same_invoice_has_positive_priced_rows": group.same_invoice_positive_priced_rows.gt(0).sum(),
    })
tables["history_summary"] = pd.DataFrame(history_rows)
display(tables["history_summary"])

,cohort,rows,distinct_invoices,singleton_invoice_rows,prior_same_invoice_rows_matched,prior_same_invoice_sku_rows_matched,prior_sku_rows_matched,customer_sku_comparable_rows,prior_customer_sku_rows_matched,prior_stripped_invoice_sku_rows_matched,prior_customer_sku_price_opposite_units_rows_matched,same_invoice_has_positive_priced_rows
0,any_exception,11805,5993,4229,0,0,11391,8945,7627,0,2816,400
1,C_invoices,9288,3836,2131,0,0,9102,8905,7599,0,2816,0
2,nonC_negative,1336,1336,1336,0,0,1179,0,0,0,0,0
3,zero_qty,0,0,0,0,0,0,0,0,0,0,0
4,positive_qty_zero_price,1179,819,760,0,0,1108,40,28,0,0,400
5,negative_price,2,2,2,0,0,2,0,0,0,0,0


In [21]:
note_tables = []
for scope, group in scopes.items():
    counts = group.groupby(["description_note_signals", "special_code_signal"],
                           dropna=False).size().rename("rows").reset_index()
    counts.insert(0, "scope", scope)
    note_tables.append(counts.sort_values("rows", ascending=False, kind="stable"))
tables["note_summary"] = pd.concat(note_tables, ignore_index=True)
display(tables["note_summary"].loc[tables["note_summary"]["scope"].eq("nonC_negative")])

,scope,description_note_signals,special_code_signal,rows
38,nonC_negative,blank,,862
39,nonC_negative,damage_or_disposal,,171
40,nonC_negative,stock_adjustment_or_count,,141
41,nonC_negative,unknown,,88
42,nonC_negative,set_or_unit_sale_wording,,32
43,nonC_negative,missing_or_lost,,18
44,nonC_negative,item_code_or_label_discrepancy,,14
45,nonC_negative,credit_note_wording,,3
46,nonC_negative,sample_or_promotion,,3
47,nonC_negative,damage_or_disposal|sample_or_promotion,,2


**Reading the output**

**`history_summary`** · Of 8,905 C-invoice lines with a known customer, 7,599 have an earlier positive-quantity line for the same customer and SKU. For 2,816 C lines, the earlier unit price and opposite quantity also match. The original purchase invoice is still unconfirmed.

**`note_summary`** · Of the 1,336 non-C negative-quantity lines, 862 have blank descriptions. Among the exclusive note combinations, 171 say damage or disposal only and 141 say stock adjustment or count only.

**如何解读输出**

**`history_summary`** · 在 8,905 行有客户编号的 C 发票记录中，7,599 行能找到同一客户、同一商品的较早正数量记录。其中 2,816 行还匹配相同单价及数值相等、符号相反的数量，原购买发票仍待核实。

**`note_summary`** · 1,336 行非 C 负数量记录中，862 行描述为空。按互斥的备注线索组合统计，171 行只命中损坏或报废措辞，141 行只命中库存调整或盘点措辞。

## Read the description, item code, and invoice reference

The note summary counts literal text and special-code signals by group. `blank` means there was no description to interpret; `unknown` means the text did not match a recognized note. Inspect a note and its referenced document as separate pieces of evidence. For instance, invoices `554550` and `554553` cite `C550456` and contain “incorrectly credited”. Invoice `554551` says “reverse previous adjustment”. The signs and text support an adjustment sequence. The workbook does not identify a corresponding shipment. For `BATHROOM METAL SIGN`, the two raw descriptions differ in trailing whitespace. The name lookup trims whitespace only to find the example and retains both stock codes. Similar descriptions cannot establish an SKU alias.

For example, the description for invoice 554143 says “damages”, while 573482 says only “check”. The first is evidence of a damage notation. The second leaves the result of that check unknown. On the zero-price rows, 1,179 have positive quantity. Among those, 40 have a customer ID and 1,139 do not. Distinguishing gifts from stock entries or a missing charge requires further documentation.

## 核查原始描述、商品编码和备注引用的发票

备注汇总按组统计字面文字和特殊商品码信号。`blank` 表示没有描述可供解释；`unknown` 表示文本未命中已设置的备注规则。备注内容和它指向的单据可以分别核查。例如 `554550` 与 `554553` 的描述包含 “incorrectly credited”，并提及 `C550456`；`554551` 则写有 “reverse previous adjustment”。正负号和描述共同支持存在调整的解释，表内没有标明对应的发货记录。`BATHROOM METAL SIGN` 的两条原始描述在末尾空格上存在差别。名称查找只为定位示例去除首尾空格，同时保留两个商品编码。仅凭名称相似无法确定两个编码互为别名。

例如，发票 554143 备注为 “damages”，说明经办记录提到了损坏；573482 只写 “check”，既未说明盘点结果也未说明客户退货。零价行中有 1,179 行数量为正，其中 40 行有客户编号、1,139 行无客户编号。赠品、库存调整或漏记价格等可能原因仍需其他凭证。

In [22]:
_TRACE_COLUMNS = [
    "ExcelRow", "InvoiceNo", "StockCode", "Description", "mentioned_invoice_id",
    "referenced_invoice_rows", "earlier_positive_quantity_rows",
    "earlier_positive_quantity_sku_rows", "earlier_same_sku_rows",
    "earlier_credit_or_nonpositive_rows", "earlier_credit_or_nonpositive_sku_rows",
    "example_earlier_same_sku_excel_row",
]


def mentioned_invoice_trace(raw: pd.DataFrame, flagged: pd.DataFrame) -> pd.DataFrame:
    """Trace each distinct invoice-shaped token per flagged source row, including misses.

    Match the full literal document ID in the whole input (prefix and zeros
    included). referenced_invoice_rows counts all matching raw lines. All other
    counts use strictly earlier InvoiceDate. A positive-quantity line can have a
    nonpositive price. The credit-or-nonpositive signal means a C-prefixed invoice,
    Quantity <= 0, or UnitPrice <= 0. Counts can overlap. Missing source dates or
    required SKU keys yield NA. The example is the lowest ExcelRow among strictly
    earlier exact invoice+SKU matches. An invoice match alone never links a return.
    """
    descriptions = flagged["Description"].astype("string").fillna("")
    extracted = descriptions.map({note: _mentioned_invoices(note) for note in descriptions.unique()})
    mentioned = flagged.loc[extracted.str.len().gt(0), [
        "ExcelRow", "InvoiceNo", "StockCode", "Description", "InvoiceDate",
    ]].copy()
    mentioned["mentioned_invoice_id"] = extracted[extracted.str.len().gt(0)]
    mentioned = mentioned.explode("mentioned_invoice_id")
    target_rows = raw.loc[raw["InvoiceNo"].isin(mentioned["mentioned_invoice_id"])]
    grouped = {key: group for key, group in target_rows.groupby("InvoiceNo", sort=False)}
    empty = target_rows.iloc[:0]
    records = []
    for row in mentioned.itertuples(index=False):
        lines = grouped.get(row.mentioned_invoice_id, empty)
        record = {
            "ExcelRow": row.ExcelRow, "InvoiceNo": row.InvoiceNo, "StockCode": row.StockCode,
            "Description": row.Description, "mentioned_invoice_id": row.mentioned_invoice_id,
            "referenced_invoice_rows": len(lines),
        }
        if pd.notna(row.InvoiceDate):
            earlier = lines.loc[lines["InvoiceDate"].lt(row.InvoiceDate)]
            positive = earlier["Quantity"].gt(0)
            credit_or_nonpositive = (
                earlier["InvoiceNo"].str.startswith("C", na=False)
                | earlier["Quantity"].le(0) | earlier["UnitPrice"].le(0)
            )
            record["earlier_positive_quantity_rows"] = int(positive.sum())
            record["earlier_credit_or_nonpositive_rows"] = int(credit_or_nonpositive.sum())
            if pd.notna(row.StockCode):
                sku = earlier["StockCode"].eq(row.StockCode)
                record["earlier_same_sku_rows"] = int(sku.sum())
                record["earlier_positive_quantity_sku_rows"] = int((positive & sku).sum())
                record["earlier_credit_or_nonpositive_sku_rows"] = int((credit_or_nonpositive & sku).sum())
                record["example_earlier_same_sku_excel_row"] = earlier.loc[sku, "ExcelRow"].min()
        records.append(record)
    trace = pd.DataFrame.from_records(records, columns=_TRACE_COLUMNS)
    return trace.astype({col: "Int64" for col in _TRACE_COLUMNS[5:]})
tables["invoice_references"] = mentioned_invoice_trace(raw, flagged)
display(tables["invoice_references"])

,ExcelRow,InvoiceNo,StockCode,Description,mentioned_invoice_id,referenced_invoice_rows,earlier_positive_quantity_rows,earlier_positive_quantity_sku_rows,earlier_same_sku_rows,earlier_credit_or_nonpositive_rows,earlier_credit_or_nonpositive_sku_rows,example_earlier_same_sku_excel_row
0,203753,554550,47566B,incorrectly credited C550456 see 47,C550456,5,0,0,1,5,1,160144
1,203763,554553,47566B,incorrectly credited C550456 see 47,C550456,5,0,0,1,5,1,160144


In [23]:
def sku_name_evidence(raw: pd.DataFrame, name: str = "BATHROOM METAL SIGN") -> pd.DataFrame:
    """One row per exact raw (Description, StockCode) in a case-sensitive trimmed lookup.

    Trimming only selects examples. Grouping retains original label whitespace,
    capitalization and exact SKU. Counts include all workbook lines. first_excel_row
    lets the reader inspect each original label and source SKU on the worksheet.
    No item map is changed.
    """
    descriptions = raw["Description"].astype("string")
    rows = raw.loc[descriptions.str.strip().eq(name.strip()).fillna(False)]
    result = (
        rows.groupby(["Description", "StockCode"], sort=True, dropna=False)
        .agg(workbook_rows=("ExcelRow", "size"), first_excel_row=("ExcelRow", "min"))
        .reset_index()
    )
    result["exact_description_length"] = result["Description"].astype("string").str.len()
    result["distinct_skus_for_exact_description"] = (
        result.groupby("Description", dropna=False)["StockCode"].transform("nunique")
    )
    result["distinct_skus_for_trimmed_name"] = rows["StockCode"].nunique()
    return result
tables["sku_names"] = sku_name_evidence(raw)
display(tables["sku_names"])

,Description,StockCode,workbook_rows,first_excel_row,exact_description_length,distinct_skus_for_exact_description,distinct_skus_for_trimmed_name
0,BATHROOM METAL SIGN,82580,641,343,19,1,2
1,BATHROOM METAL SIGN,21171,73,13319,20,1,2


**Reading the output**

**`invoice_references`** · The notes in rows 203,753 and 203,763 explicitly name C550456. Of its five earlier credit lines, row 160,144 is the sole SKU match (47566B). The reference leads to a credit document.

**`sku_names`** · Trimming whitespace makes two labels read “BATHROOM METAL SIGN”. SKU 82580 has 641 lines with a 19-character label and SKU 21171 has 73 lines with a trailing space. Keep the codes separate pending item-master evidence.

**如何解读输出**

**`invoice_references`** · 原表第 203,753 和 203,763 行的备注明确写出 C550456。该贷项发票的五行较早记录中，仅第 160,144 行的商品编码 47566B 匹配，备注追溯到的是一张贷项单据。

**`sku_names`** · 去除首尾空格后，两种标签均显示为“BATHROOM METAL SIGN”。编码 82580 有 641 行，原始标签长 19 个字符；编码 21171 有 73 行，末尾多一个空格。在获得商品主数据证据前分别保留这两个编码。

## Compare the named cases against their source rows · `case_examples`

- **A large reversal.** Compare `541431` and `C541433`. The same customer, SKU and unit price appear 16 minutes apart with quantities +74,215 and −74,215. The amounts offset. The two invoice IDs differ, even after removing the C.
- **A discount and an unpriced item.** On `C536379`, SKU `D` has the description “Discount”. A credit can refer to a discount. On `539722`, SKU `22423` has positive quantity and zero price alongside priced items. The recorded zero leaves the reason open, including whether the item was free.
- **Ledger and stock adjustments.** The code `BANK CHARGES` identifies a bank-charge entry. Compare the positive and negative prices of the `B` entries `A563185`–`A563187` (“Adjust bad debt”). These fee and debt records require different interpretations from a merchandise return. Then follow SKU `47566B` through `C550456`, `554550`, `554551`, and `554553`. Invoice `560039` explicitly mentions another code in the box; use the recorded SKU when tracing this row.

The tables show what the workbook records. Customer intent, the cause of a price of zero, physical stock movement, and settlement may require other records.

## 将指定案例与原始行对照 · `case_examples`

- **大额正负对冲。** 比较 `541431` 与 `C541433`：客户、商品编码、单价相同，相隔 16 分钟，数量分别为 +74,215 和 −74,215，两个行金额相抵。即使去掉 C，发票号也不同。
- **折扣与零价商品。** `C536379` 的商品码为 `D`，描述为 “Discount”（折扣），说明贷项可以涉及折扣。`539722` 同票中还有正常计价行，而 `22423` 数量为正、单价为零；原表留下了零价记录，是否赠品等具体原因仍待核查。
- **账务与库存调整。** 商品码 `BANK CHARGES` 标识银行费用。对照 `A563185` 至 `A563187` 的 `B` 行，查看 “Adjust bad debt”（坏账调整）所对应的正负单价。核查这类费用和坏账时应分别考虑其账务含义，不能据负号把它们当成商品退货。再沿 `47566B` 追踪 `C550456`、`554550`、`554551` 和 `554553`。`560039` 的描述明确提到箱中另一个商品编码；追踪本行时须保留原始商品编码。

表格给出工作簿记载的事实。客户意图、零价的具体原因、实际库存流向和款项结算可能需要其他单据印证。

In [24]:
examples = []
for case, invoices, sku in [
    ("same-customer opposite entries", ["541431", "C541433"], "23166"),
    ("discount credit", ["C536379"], "D"),
    ("zero recorded unit price", ["539722"], "22423"),
    ("same invoice and SKU at two prices", ["550188"], "22636"),
    ("bad-debt entries", ["A563185", "A563186", "A563187"], "B"),
    ("explicit credit note and later adjustments", ["C550456", "554550", "554551", "554553"], "47566B"),
    ("stock-code discrepancy in description", ["560039"], "20713"),
    ("damage wording in description", ["554143"], "79030G"),
    ("count/check wording in description", ["573482"], "21181"),
]:
    sample = lines.loc[lines.InvoiceNo.isin(invoices) & lines.StockCode.eq(sku)].copy()
    sample.insert(0, "case", case)
    examples.append(sample)
tables["case_examples"] = pd.concat(examples, ignore_index=True)
display(tables["case_examples"][["case", *line_fields]])
case_invoice = raw.loc[raw["InvoiceNo"].eq("539722")]
display(case_invoice.groupby("UnitPrice", as_index=False).agg(
    rows=("ExcelRow", "size"), units=("Quantity", "sum")
))
bank_fees = raw.loc[raw["StockCode"].eq("BANK CHARGES")]
display(bank_fees[["ExcelRow", "InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]].head(4))

,case,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
0,same-customer opposite entries,61621,541431,2011-01-18 10:01:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346,77183.60
1,same-customer opposite entries,61626,C541433,2011-01-18 10:17:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346,-77183.60
2,discount credit,143,C536379,2010-12-01 09:41:00,D,Discount,-1,27.50,14527,-27.50
3,zero recorded unit price,40091,539722,2010-12-21 13:45:00,22423,REGENCY CAKESTAND 3 TIER,10,0.00,14911,0.00
4,same invoice and SKU at two prices,157031,550188,2011-04-14 18:57:00,22636,CHILDS BREAKFAST SET CIRCUS PARADE,8,7.65,12457,61.20
5,same invoice and SKU at two prices,157044,550188,2011-04-14 18:57:00,22636,CHILDS BREAKFAST SET CIRCUS PARADE,1,0.00,12457,0.00
6,bad-debt entries,299984,A563185,2011-08-12 14:50:00,B,Adjust bad debt,1,11062.06,<NA>,11062.06
7,bad-debt entries,299985,A563186,2011-08-12 14:51:00,B,Adjust bad debt,1,-11062.06,<NA>,-11062.06
8,bad-debt entries,299986,A563187,2011-08-12 14:52:00,B,Adjust bad debt,1,-11062.06,<NA>,-11062.06
9,explicit credit note and later adjustments,160144,C550456,2011-04-18 13:08:00,47566B,TEA TIME PARTY BUNTING,-1300,2.55,15749,-3315.00


,UnitPrice,rows,units
0,0.00,1,10
1,0.55,2,120
2,0.85,1,24
3,1.65,2,22
4,1.95,1,10
5,2.10,4,66
6,2.95,5,48
7,3.75,1,8
8,4.25,1,6
9,4.65,1,8


,ExcelRow,InvoiceNo,StockCode,Description,Quantity,UnitPrice
4406,4408,536779,BANK CHARGES,Bank Charges,1,15.00
14435,14437,C537572,BANK CHARGES,Bank Charges,-1,95.38
28992,28994,C538680,BANK CHARGES,Bank Charges,-1,966.92
62508,62510,541505,BANK CHARGES,Bank Charges,1,15.00


## Amounts and customer activity under two definitions

Compare all original lines with lines on non-C invoices having positive quantity and positive unit price. The latter is a descriptive baseline for this calculation. It may still contain fees or other non-merchandise codes. The first table separates identified and unidentified customers and reports positive, negative, and signed line amounts. The second groups identified customers by whether they have regular entries, exceptions, or both. Recency is measured from the day after the latest workbook date. Compare days since **any activity** with days since a **regular purchase**. The signed amount is an arithmetic result, not evidence that payment was collected.

Chen, Sain and Guo study calendar 2011 and group UK transactions by postcode. They compute a line amount as quantity times price and isolate unusual customer profiles for separate examination before clustering (pp. 199–203). Our eight-column workbook has no postcode. We adapt their recency, distinct-invoice frequency and monetary measures to observed CustomerID over the workbook window and compare two definitions of activity. Among 1,563 customers with both kinds of lines, the median recency is 30 days for regular lines and 26 days when exceptional entries count as activity. Another 34 customers have exceptional entries only. These figures describe observed activity and cannot be used as cutoff-safe features for training. The paper supplies no transaction-level return or zero-price label.

## 两种统计口径下的金额和客户活动

对比全部原始行与“非 C 发票、数量为正且单价为正”的行。后者用作本次比较基准，其中仍可能含有费用或其他非商品码。第一张表区分客户编号是否缺失，分别统计正、负行金额及带符号金额合计。第二张表按是否有常规记录、异常记录或两者都有，对可识别客户分组。距最近活动天数均以工作簿最后一个日期的次日为基准，分别计算距**任意活动**和距**常规购买**的天数。带符号金额来自数量乘以单价，实际收款情况需另行核实。

Chen、Sain 与 Guo 研究 2011 自然年的英国交易，按邮编汇总，以数量乘单价计算行金额，并将异常客户画像单独审视后再聚类（第 199–203 页）。我们的八列表没有邮编，因此改用已有 CustomerID，按同一数据窗口比较距最近记录的天数、不同发票数和金额。1,563 位同时有两类记录的客户中，只看常规行时距最近记录天数的中位数为 30 天，计入特殊行后为 26 天。另有 34 位客户只出现于特殊行。这是全期描述；训练特征仍需按各自时间截点重算。论文没有为单笔记录提供退货或零价成因标签。

### How much signed value is attached to each type of line? · `amount_by_record_type`

Compute quantity × unit price in GBP on every original row, including duplicates. An exception line has a C-prefixed invoice, nonpositive quantity or nonpositive unit price. Split both regular and exception lines by whether the customer ID is known. The four row counts and signed totals add up to the whole worksheet. An invoice can contain more than one type of line, so adding the distinct-invoice counts across groups may count it twice. The negative and positive columns show how the signed total is formed. The data record no cash settlement or confirmation of a physical return.

### 不同类型明细分别涉及多少带符号金额？ · `amount_by_record_type`

保留原表全部明细（包括重复行），逐行计算数量 × 单价，单位为英镑。异常行指发票号以 C 开头、数量非正或单价非正的行。常规行和异常行再各自按有无客户编号拆分。四组的行数和带符号金额可以加总回原表。由于一张发票可能同时含多类行，直接跨组相加不同发票数可能重复计数。正、负金额列分别显示带符号总额的组成。原表没有实际资金结算流水，也没有实物退货确认记录。

In [25]:
impact_lines = raw[["CustomerID", "InvoiceNo", "InvoiceDate"]].copy()
impact_lines["signed_gbp"] = raw["Quantity"] * raw["UnitPrice"]
impact_lines["positive_line_gbp"] = impact_lines["signed_gbp"].clip(lower=0)
impact_lines["negative_line_gbp"] = impact_lines["signed_gbp"].clip(upper=0)
impact_lines["record_type"] = pd.Categorical.from_codes(
    exception.astype(int), categories=["regular", "exception"],
)
impact_lines["customer_id_status"] = pd.Categorical.from_codes(
    impact_lines["CustomerID"].isna().astype(int), categories=["identified", "missing"],
)

tables["amount_by_record_type"] = impact_lines.groupby(
    ["record_type", "customer_id_status"], observed=False,
).agg(
    rows=("InvoiceNo", "size"),
    distinct_invoices=("InvoiceNo", "nunique"),
    positive_line_gbp=("positive_line_gbp", "sum"),
    negative_line_gbp=("negative_line_gbp", "sum"),
    signed_gbp=("signed_gbp", "sum"),
).reset_index()
display(tables["amount_by_record_type"])

,record_type,customer_id_status,rows,distinct_invoices,positive_line_gbp,negative_line_gbp,signed_gbp
0,regular,identified,397884,18532,8911407.904,0.00,8911407.904
1,regular,missing,132220,1428,1755276.640,0.00,1755276.640
2,exception,identified,8945,3688,0.000,-611342.09,-611342.090
3,exception,missing,2860,2305,0.000,-307594.52,-307594.520


### How do invoice counts and amounts change for the same customer? · `customer_impact`

For each identified customer, calculate two descriptions side by side. One includes every recorded line. The other includes only regular lines with a non-C invoice and positive quantity and price. Count distinct invoice numbers within each description. Starting from all identified customers ensures that those with only exception lines still appear. Missing customer IDs remain in the preceding breakdown and cannot be linked to an individual profile.

### 同一客户在两种口径下，发票数和金额怎样变化？ · `customer_impact`

对每位有编号的客户，分别计算全部记录和常规记录的发票数与金额。常规记录要求发票号不以 C 开头、数量和单价均为正；每种口径均按不同发票号计数。从全部有编号客户出发，可以保留只有异常行的客户。缺少客户编号的行已经在上表单列，无法将它们归入某一位客户的画像。

In [26]:
identified_activity = impact_lines.loc[impact_lines["CustomerID"].notna()]
all_activity_by_id = identified_activity.groupby("CustomerID").agg(
    all_rows=("InvoiceNo", "size"),
    all_invoices=("InvoiceNo", "nunique"),
    last_activity=("InvoiceDate", "max"),
    all_signed_gbp=("signed_gbp", "sum"),
)
regular_activity = identified_activity.loc[identified_activity["record_type"] == "regular"]
regular_activity_by_id = regular_activity.groupby("CustomerID").agg(
    regular_rows=("InvoiceNo", "size"),
    regular_invoices=("InvoiceNo", "nunique"),
    last_regular=("InvoiceDate", "max"),
    regular_gross_gbp=("signed_gbp", "sum"),
)
customer_comparison = all_activity_by_id.join(regular_activity_by_id)
for column in ["regular_rows", "regular_invoices"]:
    customer_comparison[column] = customer_comparison[column].fillna(0).astype("int64")
customer_comparison["regular_gross_gbp"] = customer_comparison["regular_gross_gbp"].fillna(0.0)
customer_comparison["exception_rows"] = customer_comparison["all_rows"] - customer_comparison["regular_rows"]
customer_comparison["amount_difference_gbp"] = customer_comparison["all_signed_gbp"] - customer_comparison["regular_gross_gbp"]

### Recency and the customer-level extract

Measure recency in calendar days from the last observed date under each definition to the day after the final date in the workbook. Retain a missing regular recency when a customer has no regular purchase. The preview includes customer 12346 and one member of each cohort. For 12346, one regular invoice contributes £77,183.60, while the signed arithmetic across two invoices totals £0.00. The exported customer table contains every identified customer. This full-window comparison describes the workbook. Computing predictive features for a model requires a separate cutoff for each query.

### 距最近记录天数与逐客户明细

用工作簿最终记录日期的次日作为共同参照日，分别从每位客户最近一笔常规记录及最近一笔任意记录的日期起计算相隔的日历天数。客户没有常规购买时，相应天数保留为空。下方预览客户 12346 和每个组中的一位客户。12346 的一张常规发票金额为 £77,183.60，两张发票的带符号金额相加为 £0.00。导出的客户表包含所有有编号客户。这是对完整观察期的描述；为模型生成预测特征时，需要使用每个查询各自的历史截止点。

In [27]:
recency_reference = raw["InvoiceDate"].dt.normalize().max() + pd.Timedelta(days=1)
customer_comparison["recency_reference_date"] = recency_reference
for date_column, days_column in [
    ("last_regular", "regular_recency_days"), ("last_activity", "all_recency_days"),
]:
    last_date = customer_comparison.pop(date_column).dt.normalize()
    customer_comparison[days_column] = (recency_reference - last_date).dt.days.astype("Int64")

impact_cohort = pd.Series("mixed", index=customer_comparison.index, dtype="string")
impact_cohort.loc[customer_comparison["exception_rows"] == 0] = "regular_only"
impact_cohort.loc[customer_comparison["regular_rows"] == 0] = "exception_only"
customer_comparison["cohort"] = pd.Categorical(
    impact_cohort, categories=["regular_only", "mixed", "exception_only"],
)
tables["customer_impact"] = customer_comparison.reset_index()
sample_ids = customer_comparison.groupby("cohort", observed=True).head(1).index
display(tables["customer_impact"].loc[tables["customer_impact"]["CustomerID"].isin([*sample_ids, "12346"])])

,CustomerID,all_rows,all_invoices,all_signed_gbp,regular_rows,regular_invoices,regular_gross_gbp,exception_rows,amount_difference_gbp,recency_reference_date,regular_recency_days,all_recency_days,cohort
0,12346,2,2,0.0,1,1,77183.6,1,-77183.6,2011-12-10,326,326,mixed
1,12347,182,7,4310.0,182,7,4310.0,0,0.0,2011-12-10,3,3,regular_only
125,12503,1,1,-1126.0,0,0,0.0,1,-1126.0,2011-12-10,<NA>,338,exception_only


### Which customers are affected? · `customer_impact_summary`

Read the three groups separately. For the 1,563 customers with both regular and exception lines, the median recency is 30 days under regular purchases and 26 days under all activity. The 34 customers with only exceptions have no regular-purchase recency. A group's difference in amounts is its signed worksheet total minus its gross amount on regular lines. No customer segmentation thresholds or model training are used in this comparison.

### 哪些客户受到影响？ · `customer_impact_summary`

分三组查看结果。1,563 位同时有常规行与异常行的客户，按常规购买计算的距最近天数中位数为 30 天，按所有活动计算为 26 天。另有 34 位客户只有异常行，因而没有最近常规购买日期。每组的金额差等于全部行的带符号金额减去常规行金额。本次比较没有设置客户分层阈值，也没有训练模型。

In [28]:
tables["customer_impact_summary"] = customer_comparison.groupby("cohort", observed=False).agg(
    customers=("all_rows", "size"),
    regular_gross_gbp=("regular_gross_gbp", "sum"),
    all_signed_gbp=("all_signed_gbp", "sum"),
    amount_difference_gbp=("amount_difference_gbp", "sum"),
    median_regular_recency_days=("regular_recency_days", "median"),
    median_all_recency_days=("all_recency_days", "median"),
).reset_index()
display(tables["customer_impact_summary"])

,cohort,customers,regular_gross_gbp,all_signed_gbp,amount_difference_gbp,median_regular_recency_days,median_all_recency_days
0,regular_only,2775,2285576.522,2285576.522,0.00,65.0,65.0
1,mixed,1563,6625831.382,6024900.532,-600930.85,30.0,26.0
2,exception_only,34,0.000,-10411.240,-10411.24,<NA>,346.5


**Reading the output**

**`amount_by_record_type`** · Exception lines sum to −£611,342.09 for identified customers and −£307,594.52 where customer ID is missing. These amounts use quantity × unit price and describe ledger entries. Payment and refund settlement are unobserved.

**`customer_impact`** · Comparing each mixed customer's regular lines with all their lines, 1,557 of 1,563 customers gain at least one distinct invoice and 444 have a shorter time since the last record. Both measures are descriptive over the full workbook, with recency measured at 10 December 2011.

**`customer_impact_summary`** · Among 4,372 identified customers, 2,775 have only regular lines, 1,563 have both kinds and 34 have only exceptions. In the mixed group, counting exceptions reduces median days since the last record from 30 to 26.

**如何解读输出**

**`amount_by_record_type`** · 异常行按数量乘单价汇总，有客户编号的部分为 −£611,342.09，缺编号的部分为 −£307,594.52。这是原表记录金额，付款或退款是否到账无法由该表核实。

**`customer_impact`** · 逐人比较仅含常规行与计入全部记录的口径，1,563 位两类都有的客户中，1,557 位的不同发票数增加，444 位距最近记录的天数缩短。这是全期描述性统计，最近性以 2011 年 12 月 10 日为基准。

**`customer_impact_summary`** · 4,372 位已知客户中，2,775 位只有常规行，1,563 位两类都有，34 位只有异常行。对两类都有的客户，计入异常行后，距最近记录的天数中位数从 30 天降至 26 天。

## Build the worksheet guide and check the export order

The guide lists the other 16 sheets. Check their names against the tables constructed above. The exported detail tabs retain every qualifying row, including those beyond the notebook previews.

## 生成工作表目录并核对导出顺序

目录介绍其余 16 张表。把目录中的名称与上方代码生成的表逐一核对后再导出。明细表保留所有命中行，包含 notebook 预览以外的行。

In [29]:
guide = [
    ("sign_combinations", "Disjoint cells by C prefix, quantity sign, price sign and customer ID presence. Counts cover every raw line.", "按 C 前缀、数量及价格符号、客户编号有无交叉分组。各组合互斥，合计覆盖原表。"),
    ("missingness", "Missing fields, affected invoice counts, and intersection with missing customer IDs on the original sheet.", "原表各字段缺失行数、涉及的不同发票数及与缺客户编号的交集。"),
    ("customer_id_invoices", "Full invoice lookup for each invoice with a missing customer ID, including counts of known IDs elsewhere on the invoice.", "逐张回查含缺失客户编号的发票，统计同票其他行出现的不同已知客户编号。"),
    ("filter_attribution", "Attribute each row in a scope to the first applicable purchase filter or to retained. Scopes overlap. This is a count simulation.", "对每个筛选范围的行，统计最先命中的购买过滤规则或保留数。范围可重叠，仅作计数模拟。"),
    ("C_invoices", "All lines on C-prefixed invoices, including discounts and other non-merchandise lines.", "全部 C 前缀明细，含折扣和其他非商品记录。"),
    ("Q_nonpositive", "All lines with Quantity ≤ 0, including C invoices.", "数量 ≤ 0 的全部明细，包括 C 前缀记录。"),
    ("P_nonpositive", "All lines with UnitPrice ≤ 0. Filter price_sign to separate zero and negative recorded prices.", "单价 ≤ 0 的全部明细。用 price_sign 区分零价与负价。"),
    ("nonC_negative", "All non-C negative quantities. ExcelRow counts the header as row 1. Original descriptions and IDs are preserved.", "全部非 C 负数量明细。ExcelRow 对应原表行号，表头为第 1 行。保留原描述与编号。"),
    ("history_summary", "Counts of flagged rows with at least one match. Earlier means strictly before the flagged timestamp. Positive means Quantity > 0. Unknown customer keys are blank in row-level evidence.", "汇总至少有一条匹配的明细数。历史匹配严格早于当前时间。正数量为 Quantity > 0。明细中客户匹配留空表示无法比较。"),
    ("note_summary", "Counts by exact combination of note/code clues in each scope. Unknown means no recognized wording. Scopes overlap.", "按备注与特殊编码线索的组合计数。unknown 表示未识别到关键词。不同筛选范围之间有重叠。"),
    ("case_examples", "Selected original entries in chronological context. Signed amount is quantity times unit price in GBP.", "从原表选取的交易及上下文。带符号金额为数量乘单价，单位英镑。"),
    ("invoice_references", "Invoice-shaped tokens in the Description and exact matches in the workbook, with earlier-line counts.", "备注里的发票号形式文本，在原表中精确回查，并统计较早的明细。"),
    ("sku_names", "Exact stored labels and codes for a name lookup after trimming surrounding whitespace. Product identity remains unverified.", "按名称去除首尾空白查找，展示原始名称和编码。商品是否完全相同仍待商品主数据核实。"),
    ("amount_by_record_type", "Signed line arithmetic by exception status and customer ID availability. Actual payments and refunds are unobserved.", "按标记与有无客户编号汇总带符号行金额。原表没有付款到账或退款流水。"),
    ("customer_impact_summary", "Identified customers grouped by whether they have regular, exceptional or both kinds of lines.", "按已知客户拥有常规行、异常行或两类兼有进行汇总。"),
    ("customer_impact", "Per identified customer, compare distinct invoices, recency and monetary totals under both definitions. Full-window descriptive analysis.", "对每一位已知客户，比较两种口径的不同发票数、距最近记录天数及金额。仅作全期描述。"),
]
tables["guide"] = pd.DataFrame(guide, columns=["Sheet", "What it contains", "内容与口径"])
sheet_order = [
    "guide", "sign_combinations", "missingness", "customer_id_invoices", "filter_attribution",
    "history_summary", "note_summary", "case_examples", "C_invoices", "Q_nonpositive",
    "P_nonpositive", "nonC_negative", "invoice_references", "sku_names",
    "amount_by_record_type", "customer_impact_summary", "customer_impact",
]
assert set(tables["guide"]["Sheet"]) == set(sheet_order) - {"guide"}
assert set(tables) == set(sheet_order) and len(tables) == 17
tables = {name: tables[name] for name in sheet_order}
display(tables["guide"])

,Sheet,What it contains,内容与口径
0,sign_combinations,"Disjoint cells by C prefix, quantity sign, pri...",按 C 前缀、数量及价格符号、客户编号有无交叉分组。各组合互斥，合计覆盖原表。
1,missingness,"Missing fields, affected invoice counts, and i...",原表各字段缺失行数、涉及的不同发票数及与缺客户编号的交集。
2,customer_id_invoices,Full invoice lookup for each invoice with a mi...,逐张回查含缺失客户编号的发票，统计同票其他行出现的不同已知客户编号。
3,filter_attribution,Attribute each row in a scope to the first app...,对每个筛选范围的行，统计最先命中的购买过滤规则或保留数。范围可重叠，仅作计数模拟。
4,C_invoices,"All lines on C-prefixed invoices, including di...",全部 C 前缀明细，含折扣和其他非商品记录。
5,Q_nonpositive,"All lines with Quantity ≤ 0, including C invoi...",数量 ≤ 0 的全部明细，包括 C 前缀记录。
6,P_nonpositive,All lines with UnitPrice ≤ 0. Filter price_sig...,单价 ≤ 0 的全部明细。用 price_sign 区分零价与负价。
7,nonC_negative,All non-C negative quantities. ExcelRow counts...,全部非 C 负数量明细。ExcelRow 对应原表行号，表头为第 1 行。保留原描述与编号。
8,history_summary,Counts of flagged rows with at least one match...,汇总至少有一条匹配的明细数。历史匹配严格早于当前时间。正数量为 Quantity > 0。明...
9,note_summary,Counts by exact combination of note/code clues...,按备注与特殊编码线索的组合计数。unknown 表示未识别到关键词。不同筛选范围之间有重叠。


## Export and sources

Write all tables to `results/transaction_exceptions.xlsx`. In Excel, filter by `ExcelRow`, `InvoiceNo`, or `StockCode` to trace a row. The original eight fields are present in the full case sheets. Chen, Sain and Guo (2012), pp. 199–203, discuss transaction preparation and RFM customer segmentation. Their analysis supplies context for the customer-activity comparison. The counts and case evidence here come directly from the identified workbook. [Chen, Sain & Guo (2012)](https://doi.org/10.1057/dbm.2012.17) · [Chen (2015), UCI Online Retail, CC BY 4.0](https://doi.org/10.24432/C5BW33)

## 导出与来源

将全部表格写入 `results/transaction_exceptions.xlsx`。在 Excel 中可用 `ExcelRow`、`InvoiceNo` 或 `StockCode` 筛选和追溯明细，完整案例工作表保留八个原始字段。Chen、Sain 和 Guo（2012，第 199–203 页）讨论了交易数据准备与基于 RFM 的客户分群，为此次客户活动口径比较提供背景。本次统计量和案例证据均取自已校验的原始工作簿。[Chen、Sain 和 Guo（2012）](https://doi.org/10.1057/dbm.2012.17) · [Chen（2015），UCI Online Retail，CC BY 4.0](https://doi.org/10.24432/C5BW33)

In [30]:
from tempfile import TemporaryDirectory
from openpyxl.utils import get_column_letter

def write_workbook(tables: dict[str, pd.DataFrame], output: Path) -> None:
    """Write filterable sheets, preserving original descriptions as literal text."""
    path = Path(output)
    if path.suffix.lower() != ".xlsx":
        raise ValueError("Choose an .xlsx output file.")
    path.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=path.parent) as work:
        temp = Path(work) / path.name
        with pd.ExcelWriter(temp, engine="openpyxl", datetime_format="yyyy-mm-dd hh:mm:ss") as writer:
            for name, frame in tables.items():
                frame.to_excel(writer, sheet_name=name, index=False)
                sheet = writer.sheets[name]
                sheet.freeze_panes = "B2"
                sheet.auto_filter.ref = sheet.dimensions
                for number, col in enumerate(frame.columns, start=1):
                    width = 48 if col == "Description" else max(17, len(col) + 3)
                    if name == "guide":
                        width = 32 if number == 1 else 105
                    sheet.column_dimensions[get_column_letter(number)].width = width
                # A leading '=' in an original description is data, not an Excel formula.
                for row in sheet.iter_rows(min_row=2):
                    for cell in row:
                        if cell.data_type == "f":
                            cell.data_type = "s"
        temp.replace(path)

write_workbook(tables, output)
print(f"Saved {len(tables)} sheets / 已保存 {len(tables)} 张工作表：{output}")

Saved 17 sheets / 已保存 17 张工作表：/workspace/online-retail-top10-recommendation/results/transaction_exceptions.xlsx
